# LLM Buggy Code Detection Ablation Notebook

This notebook evaluates different large language models (Phi-3.5, DeepSeek-Coder, CodeGemma, etc.) on buggy code detection tasks using multiple prompt styles (direct, chain-of-thought, few-shot).


In [ ]:
# Install dependencies (run once per runtime)
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q pandas scikit-learn tqdm seaborn matplotlib

In [ ]:
# Prevent CUDA memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Dec  4 22:09:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `token3` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `token3`


## Model selection
Pick which LLM you want to evaluate. The same pipeline will work for Phi-3.5, DeepSeek-Coder, and CodeGemma.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Select a model to use:")
print("1 = Phi-3.5 Mini (Instruct)")
print("2 = DeepSeek-Coder 1.3B (Instruct)")
print("3 = CodeGemma 2B")

choice = input("Enter choice (1-3): ").strip()

model_map = {
    "1": "microsoft/phi-3.5-mini-instruct",
    "2": "deepseek-ai/deepseek-coder-1.3b-instruct",
    "3": "google/codegemma-2b",
}

if choice not in model_map:
    raise ValueError("Invalid model choice.")

model_name = model_map[choice]

# Set default dtype
torch.set_default_dtype(torch.float16)

# ✅ EXTENDED CONTEXT: Set to 16K tokens
EXTENDED_MAX_LENGTH = 16384  # 16K tokens (4x the default)

# Load tokenizer with extended length
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
tokenizer.model_max_length = EXTENDED_MAX_LENGTH  # ✅ Extend tokenizer limit

print(f"Tokenizer max length set to: {tokenizer.model_max_length}")

# Load model with extended context
if "phi-3.5" in model_name:
    # Phi-3.5 supports up to 128K with RoPE scaling
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        attn_implementation="eager",
        device_map="auto",
        trust_remote_code=True,
        rope_scaling={
            "type": "linear",
            "factor": 4.0  # Scale from 4K to 16K
        },
    )
    # Update model config
    model.config.max_position_embeddings = EXTENDED_MAX_LENGTH

elif "deepseek" in model_name:
    # DeepSeek-Coder with extended context and 8-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        rope_scaling={
            "type": "linear",
            "factor": 4.0  # Scale from 4K to 16K
        },
    )
    # Update model config
    model.config.max_position_embeddings = EXTENDED_MAX_LENGTH

elif "codegemma" in model_name:
    # CodeGemma already supports 8K, can extend to 16K
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.max_position_embeddings = EXTENDED_MAX_LENGTH

model.eval()

print(f"\n⭐ Successfully loaded {model_name}")
print(f"📏 Model max position embeddings: {model.config.max_position_embeddings}")
print(f"📏 Tokenizer max length: {tokenizer.model_max_length}")

Select a model to use:
1 = Phi-3.5 Mini (Instruct)
2 = DeepSeek-Coder 1.3B (Instruct)
3 = CodeGemma 2B
Enter choice (1-3): 3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


⭐ Successfully loaded google/codegemma-2b
📏 Model max position embeddings: 16384
📏 Tokenizer max length: 16384


## Prompt builder
This supports multiple prompt styles: `direct`, `cot` (chain-of-thought), and `fewshot`.


In [ ]:
def build_prompt(code, style="direct"):

    if style == "direct":
        return f"""
Does this code have a bug? Answer only True or False, without any explanation.

<CODE>
{code}
<CODE>
"""

    if style == "cot":
        return f"""
You are an expert software engineer.
Reason step by step about whether the following c or c++ code is buggy.
At the VERY END, output exactly one word: True or False.

<CODE>
{code}
<CODE>
Answer:
"""

    if style == "fewshot":
        return f"""
Classify code as buggy (True) or clean (False). Respond with exactly one word.

static av_cold int vdadec_init(AVCodecContext *avctx)

{{

    VDADecoderContext *ctx = avctx->priv_data;

    struct vda_context *vda_ctx = &ctx->vda_ctx;

    OSStatus status;

    int ret;



    ctx->h264_initialized = 0;



    /* init pix_fmts of codec */

    if (!ff_h264_vda_decoder.pix_fmts) {{

        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;

        else

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts;

    }}



    /* init vda */

    memset(vda_ctx, 0, sizeof(struct vda_context));

    vda_ctx->width = avctx->width;

    vda_ctx->height = avctx->height;

    vda_ctx->format = 'avc1';

    vda_ctx->use_sync_decoding = 1;

    vda_ctx->use_ref_buffer = 1;

    ctx->pix_fmt = avctx->get_format(avctx, avctx->codec->pix_fmts);

    switch (ctx->pix_fmt) {{

    case AV_PIX_FMT_UYVY422:

        vda_ctx->cv_pix_fmt_type = '2vuy';

        break;

    case AV_PIX_FMT_YUYV422:

        vda_ctx->cv_pix_fmt_type = 'yuvs';

        break;

    case AV_PIX_FMT_NV12:

        vda_ctx->cv_pix_fmt_type = '420v';

        break;

    case AV_PIX_FMT_YUV420P:

        vda_ctx->cv_pix_fmt_type = 'y420';

        break;

    default:

        av_log(avctx, AV_LOG_ERROR, "Unsupported pixel format: %d\n", avctx->pix_fmt);

        goto failed;

    }}

    status = ff_vda_create_decoder(vda_ctx,

                                   avctx->extradata, avctx->extradata_size);

    if (status != kVDADecoderNoErr) {{

        av_log(avctx, AV_LOG_ERROR,

                "Failed to init VDA decoder: %d.\n", status);

        goto failed;

    }}

    avctx->hwaccel_context = vda_ctx;



    /* changes callback functions */

    avctx->get_format = get_format;

    avctx->get_buffer2 = get_buffer2;

#if FF_API_GET_BUFFER

    // force the old get_buffer to be empty

    avctx->get_buffer = NULL;

#endif



    /* init H.264 decoder */

    ret = ff_h264_decoder.init(avctx);

    if (ret < 0) {{

        av_log(avctx, AV_LOG_ERROR, "Failed to open H.264 decoder.\n");

        goto failed;

    }}

    ctx->h264_initialized = 1;



    return 0;



failed:

    vdadec_close(avctx);

    return -1;

}}

Label: False

Example 2:
Code:
int divide(int a, int b) {{ return a / 0;; }}
Label: True

Now classify this c or c++ code:
<CODE>
{code}
<CODE>

Label:
"""
    if style == "neurosym":
      return f"""
Determine if the following c or c++ code is buggy.
Reason step by step about whether the following c or c++ code is buggy.
Find and consider symbolic features in the code, including number of nodes,
functions, if statements, loops, and features like complexity, nesting depth,
and suspicous patterns in the code when classifying.
You MUST answer with exactly one word: True or False.
True means the code is buggy. False means the code is clean.

<CODE>
{code}
<CODE>
"""

    raise ValueError(f"Unknown prompt style: {style}")


##Evaluating using CodeXGlue C/C++ code snippets


In [ ]:
def parse_llm_output(raw_out: str):
    """
    Robust classifier for LLM outputs without relying on last-word parsing.
    Steps added:
      - Extracts ONLY the portion after the *second* <CODE> or <code> marker.
      - Then applies your semantic pattern detection.

    Returns:
        1  = buggy
        0  = clean
       -1  = invalid
    """

    # ----------------------------------------------------------------------
    # 0. Handle empty output
    # ----------------------------------------------------------------------
    if not raw_out or raw_out.strip() == "":
        return -1, "[EMPTY]"

    text = raw_out

    # ----------------------------------------------------------------------
    # 1. Extract only the content AFTER the second <CODE> / <code>
    # ----------------------------------------------------------------------
    # Uppercase
    parts = text.split("<CODE>")
    if len(parts) >= 3:
        text = parts[-1].strip()

    # Lowercase (DeepSeek sometimes uses lowercase)
    parts_lower = text.split("<code>")
    if len(parts_lower) >= 3:
        text = parts_lower[-1].strip()

    if not text:
        return -1, "[EMPTY_AFTER_CODE]"

    # Normalize for pattern matching
    text = text.lower().strip()

    # ----------------------------------------------------------------------
    # 2. Exact True/False
    # ----------------------------------------------------------------------
    if text == "true":
        return 1, "true"
    if text == "false":
        return 0, "false"

    # ----------------------------------------------------------------------
    # 3. Semantic phrase detection (FULL-OUTPUT SCAN)
    # ----------------------------------------------------------------------
    buggy_patterns = [
        "is buggy",
        "contains a bug",
        "has a bug",
        "not clean",
        "is not clean",
        "causes an error",
        "unsafe",
        "incorrect behavior",
        "logic error",
        "is incorrect",
        "is not correct",
        "memory bug",
    ]

    clean_patterns = [
        "clean code",
        "is clean",
        "no bug",
        "not buggy",
        "seems correct",
        "is correct",
        "is safe",
        "does not contain a bug",
        "bug free",
        "safe code",
        "not a bug",
        "correct behavior",
        "seems to be correct",
    ]

    # BUGGY → higher priority
    for p in buggy_patterns:
        if p in text:
            return 1, p

    # CLEAN → second priority
    for p in clean_patterns:
        if p in text:
            return 0, p

    # ----------------------------------------------------------------------
    # 4. Fallback to any standalone occurrence of true/false
    # ----------------------------------------------------------------------
    import re
    matches = re.findall(r"\b(true|false)\b", text)
    if matches:
        last = matches[-1]
        return (1 if last == "true" else 0), last

    # ----------------------------------------------------------------------
    # 5. Still no signal → INVALID
    # ----------------------------------------------------------------------
    return -1, "[INVALID]"


In [ ]:
def evaluate_codexglue_with_csv(prompt_style="direct"):
    print(f"\n=== Running CodeXGLUE Evaluation ({prompt_style}) ===")

    rows = []
    y_true = []
    y_pred = []

    max_gen_tokens = 300

    MODEL_MAX_LENGTH = getattr(model.config, 'max_position_embeddings', 4096)
    MAX_INPUT_LENGTH = MODEL_MAX_LENGTH - max_gen_tokens - 50

    print(f"Model max context: {MODEL_MAX_LENGTH}")
    print(f"Using max_new_tokens={max_gen_tokens}")
    print(f"Max input length: {MAX_INPUT_LENGTH}")

    for idx, example in enumerate(tqdm(codex_ds, total=len(codex_ds), desc=f"CodeXGLUE — {prompt_style}")):

        code = get_codex_code(example)
        true_label = get_codex_label(example)
        prompt = build_prompt(code, style=prompt_style)

        # Tokenize
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        ).to(device)

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_gen_tokens,
                min_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # FULL decoded model output
        full_decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract generated region
        prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
        raw_out = full_decoded[len(prompt_text):].strip()

        # Parse via your improved output parser
        pred, clean_fragment = parse_llm_output(raw_out)

        # ------- LOGGING ADDED BACK -------
        if idx % 10 == 0:  # log every 5th sample or invalids
            print("\n---------------- SAMPLE LOG ----------------")
            print(f"Index: {idx}")
            print("PROMPT:\n", prompt[:350], "...")
            print("\nRAW MODEL OUTPUT:\n", raw_out)
            print(f"PARSED SIGNAL → pred={pred}, fragment='{clean_fragment}'")
            print("------------------------------------------------\n")
        # -----------------------------------

        y_true.append(true_label)
        y_pred.append(pred)

        rows.append({
    "dataset": "CodeXGLUE",
    "language": "C/C++",
    "prompt_style": prompt_style,
    "model_name": model_name,

    # Code + prompt
    "code_snippet": code[:200] + "...",
    "prompt": prompt[:500] + "...",

    "full_model_output": full_decoded,

    # Existing: only the generated part
    "raw_model_output": raw_out if raw_out else "[EMPTY]",

    # Parse signal
    "cleaned_signal": clean_fragment,

    # Labels
    "true_label_num": true_label,
    "pred_label_num": pred,
    "pred_label_str": (
        "Buggy (1)" if pred == 1 else
        "Clean (0)" if pred == 0 else
        "INVALID"
    ),
})


    # Convert invalids → clean
    y_pred_eval = [0 if p == -1 else p for p in y_pred]

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred_eval)
    precision = precision_score(y_true, y_pred_eval, zero_division=0)
    recall = recall_score(y_true, y_pred_eval, zero_division=0)
    f1 = f1_score(y_true, y_pred_eval, zero_division=0)

    print(f"\n===== CodeXGLUE — {prompt_style} — {model_name} =====")
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print(f"Invalid outputs: {y_pred.count(-1)}/{len(y_pred)}")

    df = pd.DataFrame(rows)
    filename = f"codexglue_{prompt_style}_{model_name.split('/')[-1]}.csv"
    df.to_csv(filename, index=False)
    print(f"Saved predictions to {filename}")

    return df


In [ ]:
from datasets import load_dataset

# Load 100 examples from the test split
print("Loading CodeXGLUE Defect Detection Dataset...")
codex_ds = load_dataset("google/code_x_glue_cc_defect_detection", split="test[:50]")

# CodeXGLUE helpers
def get_codex_code(example):
    return example["func"]

def get_codex_label(example):
    return int(example["target"])

Loading CodeXGLUE Defect Detection Dataset...


In [ ]:
codexglue_results = {}
prompt_styles = ["neurosym"]
for style in prompt_styles:
    df = evaluate_codexglue_with_csv(prompt_style=style)
    codexglue_results[style] = df

print("\nFinished CodeXGLUE Ablation.")


=== Running CodeXGLUE Evaluation (neurosym) ===
Model max context: 16384
Using max_new_tokens=300
Max input length: 16034


CodeXGLUE — neurosym:   2%|▏         | 1/50 [00:10<08:46, 10.74s/it]


---------------- SAMPLE LOG ----------------
Index: 0
PROMPT:
 
Determine if the following c or c++ code is buggy.
Reason step by step about whether the following c or c++ code is buggy.
Find and consider symbolic features in the code, including number of nodes,
functions, if statements, loops, and features like complexity, nesting depth,
and suspicous patterns in the code when classifying.
You MUST answer wit ...

RAW MODEL OUTPUT:
 int ff_get_extradata(AVCodecContext *s, AVIOContext *pb, int size)

{

    int i, j, k;

    int64_t *extradata = NULL;

    int64_t *extradata_end = NULL;

    int64_t *extradata_cur = NULL;

    int64_t *extradata_next = NULL;

    int64_t *extradata_next_end = NULL;

    int64_t *extradata_next_cur = NULL;

    int64_t *extradata_next_next = NULL;

    int64_t *extradata_next_next_end = NULL;

    int64_t *extradata_next_next_cur = NULL;

    int64_t *extradata_next_next_next = NULL;

    int64_t *extradata_next_next_next_end = NULL;

    int64_t *extr

CodeXGLUE — neurosym:  22%|██▏       | 11/50 [00:23<00:33,  1.17it/s]


---------------- SAMPLE LOG ----------------
Index: 10
PROMPT:
 
Determine if the following c or c++ code is buggy.
Reason step by step about whether the following c or c++ code is buggy.
Find and consider symbolic features in the code, including number of nodes,
functions, if statements, loops, and features like complexity, nesting depth,
and suspicous patterns in the code when classifying.
You MUST answer wit ...

RAW MODEL OUTPUT:
 <|file_separator|>
PARSED SIGNAL → pred=-1, fragment='[INVALID]'
------------------------------------------------



...


---------------- SAMPLE LOG ----------------
Index: 30
PROMPT:
 
Determine if the following c or c++ code is buggy.
Reason step by step about whether the following c or c++ code is buggy.
Find and consider symbolic features in the code, including number of nodes,
functions, if statements, loops, and features like complexity, nesting depth,
and suspicous patterns in the code when classifying.
You MUST answer wit ...

RAW MODEL OUTPUT:
 <|file_separator|>
PARSED SIGNAL → pred=-1, fragment='[INVALID]'
------------------------------------------------



CodeXGLUE — neurosym:  84%|████████▍ | 42/50 [01:27<00:12,  1.60s/it]


---------------- SAMPLE LOG ----------------
Index: 40
PROMPT:
 
Determine if the following c or c++ code is buggy.
Reason step by step about whether the following c or c++ code is buggy.
Find and consider symbolic features in the code, including number of nodes,
functions, if statements, loops, and features like complexity, nesting depth,
and suspicous patterns in the code when classifying.
You MUST answer wit ...

RAW MODEL OUTPUT:
 <|file_separator|>
PARSED SIGNAL → pred=-1, fragment='[INVALID]'
------------------------------------------------



CodeXGLUE — neurosym: 100%|██████████| 50/50 [01:38<00:00,  1.97s/it]


===== CodeXGLUE — neurosym — google/codegemma-2b =====
Accuracy : 0.4
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
Invalid outputs: 50/50
Saved predictions to codexglue_neurosym_codegemma-2b.csv

Finished CodeXGLUE Ablation.
